# HGSEL Phase 4 — Kaggle GPU Gate Run

**Purpose:** Clear all three Phase 4 strict gates on real multi-GPU hardware (2× T4, NCCL).

**Gates to clear:**

1. **GPU Baseline** — CUDA device, representative `bench_steps`, throughput + memory metrics

2. **DDP Parity** — `world_size=2`, NCCL backend, CUDA device, HGSEL convergence  

3. **Comm Microbenchmark** — `world_size=2`, NCCL backend, CUDA device, `comm_share < 20%`

**Expected runtime:** ~25–40 minutes total on 2× T4.

**Setup:** Enable GPU accelerator in Kaggle → Settings → Accelerator → GPU T4 x2.

In [ ]:
import subprocess, json, os, sys, textwrap, shutil
from pathlib import Path

# ── Clone repo & set working directory ────────────────────────────────────────

REPO_URL = "https://github.com/toxzak-svg/hgsel-moe.git"
KAGGLE_WORK_DIR = Path("/kaggle/working/hgsel-moe")
LOCAL_WORK_DIR = Path.cwd()  # Use current directory if already in repo

# Detect if we're already in the repository (local execution)
def is_in_repo(path: Path) -> bool:
    """Check if path contains the hgsel package and key files."""
    return (path / "hgsel").exists() and (path / "experiments").exists()

# Determine working directory
if is_in_repo(LOCAL_WORK_DIR):
    # Local execution: use current directory
    WORK_DIR = LOCAL_WORK_DIR
    print(f"✓ Local repository detected at: {WORK_DIR}")
    print("✓ Skipping git clone (using local files)")
elif KAGGLE_WORK_DIR.exists() and is_in_repo(KAGGLE_WORK_DIR):
    # Kaggle with existing repo
    WORK_DIR = KAGGLE_WORK_DIR
    print(f"✓ Existing Kaggle repo at: {WORK_DIR}")
    print("Repo already present, pulling latest...")
    result = subprocess.run(
        ["git", "-C", str(WORK_DIR), "pull", "--ff-only"],
        capture_output=True, text=True
    )
    print(result.stdout)
else:
    # Need to clone (fresh Kaggle environment)
    WORK_DIR = KAGGLE_WORK_DIR
    
    # Force recloning if previous attempt failed or directory is corrupted
    FORCE_RECLONE = True  # Set to False if you want to keep existing repo
    
    if WORK_DIR.exists() and FORCE_RECLONE:
        print("Removing existing directory for fresh clone...")
        shutil.rmtree(WORK_DIR)
    
    if not WORK_DIR.exists():
        print("Cloning repo...")
        # Try multiple times in case of transient network issues
        max_retries = 3
        for attempt in range(max_retries):
            result = subprocess.run(
                ["git", "clone", "--depth=1", "--single-branch", REPO_URL, str(WORK_DIR)],
                capture_output=True, text=True
            )
            if result.returncode == 0:
                print(result.stdout)
                print("✓ Clone complete")
                break
            else:
                print(f"Attempt {attempt + 1}/{max_retries} failed")
                print("STDERR:", result.stderr)
                if attempt < max_retries - 1:
                    print("Retrying in 3 seconds...")
                    import time
                    time.sleep(3)
                else:
                    # Try alternative: clone without --depth=1
                    print("Trying full clone without --depth=1...")
                    result = subprocess.run(
                        ["git", "clone", REPO_URL, str(WORK_DIR)],
                        capture_output=True, text=True
                    )
                    if result.returncode == 0:
                        print(result.stdout)
                        print("✓ Clone complete (full)")
                    else:
                        print("STDERR:", result.stderr)
                        raise RuntimeError("git clone failed after all retries")

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
print(f"CWD: {os.getcwd()}")

# ── Install extra dependencies not bundled with Kaggle's PyTorch image ─────────

print("\nInstalling extra packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "wandb>=0.13.0", "tensorboard>=2.8.0"],
    check=True
)
print("✓ Dependencies ready")

# ── Create result directories ──────────────────────────────────────────────────

for d in ["results/gpu_baseline", "results/phase4", "results/token_exchange_micro"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("✓ Result directories created")

In [ ]:
# ── GPU inventory ──────────────────────────────────────────────────────────────

import torch

n_gpus = torch.cuda.device_count()
print(f"PyTorch {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  {props.total_memory / 1024**3:.1f} GB VRAM")

if n_gpus < 2:
    print("\n⚠  Only 1 GPU visible. DDP + microbenchmark will fall back to world_size=1.")
    print("   → In Kaggle: Settings → Accelerator → GPU T4 x2")

NPROC = min(n_gpus, 2) if n_gpus > 0 else 1
print(f"\nUsing nproc_per_node = {NPROC}")

In [ ]:
# ── Helper: run a shell command, stream output, raise on failure ───────────────

def run_cmd(cmd: list[str], label: str = "") -> int:
    """Run cmd, print live output, return exit code."""
    tag = f"[{label}] " if label else ""
    print(f"{tag}Running: {' '.join(cmd)}\n{'─'*72}")
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=str(WORK_DIR)
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    status = "✓ Done" if proc.returncode == 0 else f"✗ Exit {proc.returncode}"
    print(f"\n{'─'*72}\n{tag}{status}\n")
    return proc.returncode


def load_json(path: str) -> dict:
    with open(path) as f:
        return json.load(f)


def pp(data: dict, indent: int = 2):
    print(json.dumps(data, indent=indent, default=str))

## Gate 1 — GPU Baseline

Single-device CUDA run with production-representative `d_model=512`, `d_ff=2048`, `bench_steps=50`.  
Establishes throughput, forward/backward timing, peak memory, and expert utilisation histogram.

In [ ]:
BASELINE_JSON = "results/gpu_baseline/train_gpu_baseline.json"

# T4-safe config: float16, 512-dim model, 50 bench steps
baseline_cmd = [
    sys.executable, "experiments/train_gpu_baseline.py",
    "--device",       "cuda",
    "--dtype",        "float16",
    "--models",       "dense,hgsel",
    "--d-model",      "512",
    "--d-ff",         "2048",
    "--num-layers",   "4",
    "--num-heads",    "8",
    "--num-experts",  "64",
    "--batch-size",   "16",
    "--seq-length",   "128",
    "--warmup-steps", "10",
    "--bench-steps",  "50",
    "--output",       BASELINE_JSON,
]

rc = run_cmd(baseline_cmd, label="BASELINE")
assert rc == 0, f"Baseline script exited with code {rc}"

# ── Display baseline key metrics ───────────────────────────────────────────────
baseline_data = load_json(BASELINE_JSON)

print(f"{'Model':<10} {'Device':<6} {'tokens/sec':>12} {'fwd_p50 ms':>12} {'bwd_p50 ms':>12} {'peak_mem MB':>12}")
print("─" * 68)
for r in baseline_data.get("results", []):
    model   = r.get("model_kind", "?")
    device  = r.get("device", "?")
    tps     = r.get("throughput", {}).get("tokens_per_sec", float("nan"))
    fwd     = r.get("timings_ms", {}).get("forward_p50", float("nan"))
    bwd     = r.get("timings_ms", {}).get("backward_p50", float("nan"))
    mem     = r.get("memory_mb", {}).get("peak_allocated", float("nan"))
    print(f"{model:<10} {device:<6} {tps:>12,.0f} {fwd:>12.2f} {bwd:>12.2f} {mem:>12.1f}")

print()
hgsel_r = next((r for r in baseline_data.get("results", []) if r.get("model_kind") == "hgsel"), None)
if hgsel_r:
    util = hgsel_r.get("expert_utilization", {})
    if util:
        entropy = util.get("entropy_mean", "n/a")
        print(f"HGSEL expert entropy (load balance): {entropy}")
        hist = util.get("histogram", [])
        if hist:
            print(f"Expert load histogram (normalised): min={min(hist):.4f}  max={max(hist):.4f}")

## Gate 2 — DDP Parity (2× GPU, NCCL)

Runs `train_distributed_300m.py` with `torchrun --nproc_per_node=2`, NCCL backend, CUDA.  
Confirms: gradient sync works, global batch math is correct, HGSEL training converges across ranks.

In [ ]:
PARITY_JSON = "results/phase4/ddp_parity.json"

# torchrun with 2 GPUs, NCCL, HGSEL enabled
# Use a small-but-not-trivial model: 256-dim, 2 layers, 3 epochs, 200 batches
parity_cmd = [
    "torchrun",
    f"--nproc_per_node={NPROC}",
    "--rdzv_backend=c10d",
    "--rdzv_endpoint=localhost:29500",
    "experiments/train_distributed_300m.py",
    "--use-hgsel",
    "--backend",     "nccl" if NPROC > 1 else "gloo",
    "--d-model",     "256",
    "--d-ff",        "1024",
    "--num-layers",  "2",
    "--num-heads",   "4",
    "--num-experts", "32",
    "--batch-size",  "16",
    "--seq-length",  "128",
    "--num-batches", "200",
    "--num-epochs",  "3",
    "--results-json", PARITY_JSON,
]

rc = run_cmd(parity_cmd, label="DDP-PARITY")
assert rc == 0, f"DDP parity script exited with code {rc}"

# ── Display DDP parity results ─────────────────────────────────────────────────
parity_data = load_json(PARITY_JSON)
meta = parity_data.get("metadata", {})
results = parity_data.get("results", {})

print(f"world_size      : {meta.get('world_size', '?')}")
print(f"backend         : {meta.get('backend', '?')}")
print(f"device          : {meta.get('device', '?')}")
print(f"global_batch    : {meta.get('global_batch_size', '?')}")
print()

train_losses = results.get("train_loss", [])
val_losses   = results.get("val_loss", [])
perplexities = results.get("val_perplexity", [])

print(f"{'Epoch':<6} {'train_loss':>11} {'val_loss':>10} {'perplexity':>12}")
print("─" * 44)
for i, (tl, vl, pp_) in enumerate(zip(train_losses, val_losses, perplexities), 1):
    print(f"{i:<6} {tl:>11.4f} {vl:>10.4f} {pp_:>12.2f}")

if len(train_losses) > 1:
    delta = train_losses[-1] - train_losses[0]
    trend = "✓ decreasing" if delta < 0 else "✗ NOT decreasing"
    print(f"\nTrain loss trend: {delta:+.4f}  {trend}")

## Gate 3 — Expert-Parallel Communication Microbenchmark (2× GPU, NCCL)

Runs `benchmark_token_exchange_micro.py` with `torchrun`, sweeping production `hidden_dim` values.

**Decision thresholds:**

- `comm_share_p50 < 0.20` → **PASS** (expert sharding is viable)
- `comm_share_p50 0.20–0.40` → **WARN** (optimise dispatch before scaling claims)
- `comm_share_p50 > 0.40` → **STOP** (redesign all-to-all path)

In [ ]:
MICRO_JSON = "results/token_exchange_micro/benchmark_token_exchange_micro.json"

# Production-representative sweep: tokens_per_rank and hidden_dims matter most
# T4 is PCIe (pessimistic interconnect) — a good stress test for the comm gate
micro_cmd = [
    "torchrun",
    f"--nproc_per_node={NPROC}",
    "--rdzv_backend=c10d",
    "--rdzv_endpoint=localhost:29501",
    "experiments/benchmark_token_exchange_micro.py",
    "--tokens-per-rank", "2048,4096",
    "--hidden-dims",     "512,1024",
    "--routing-modes",   "balanced,moderate_skew,worst_skew",
    "--dtype",           "float16",
    "--device",          "cuda",
    "--backend",         "nccl" if NPROC > 1 else "gloo",
    "--warmup-iters",    "10",
    "--bench-iters",     "50",
    "--output",          MICRO_JSON,
]

rc = run_cmd(micro_cmd, label="MICROBENCH")
assert rc == 0, f"Microbenchmark script exited with code {rc}"

# ── Display microbenchmark comm/compute ratios ─────────────────────────────────
micro_data = load_json(MICRO_JSON)
meta_m = micro_data.get("metadata", {})
print(f"world_size : {meta_m.get('world_size', '?')}  backend : {meta_m.get('backend', '?')}  device : {meta_m.get('device', '?')}")
print(f"thresholds : warn={meta_m.get('threshold_warn', 0.20):.0%}  stop={meta_m.get('threshold_stop', 0.40):.0%}")
print()

WARN = meta_m.get("threshold_warn", 0.20)
STOP = meta_m.get("threshold_stop", 0.40)

def decision_str(share: float) -> str:
    if share > STOP:  return "STOP ✗"
    if share > WARN:  return "WARN ⚠"
    return "PASS ✓"

print(f"{'hidden_dim':>10} {'tokens/rank':>12} {'routing':>15} {'comm_p50':>10} {'comm_p95':>10} {'decision':>12}")
print("─" * 74)
for entry in micro_data.get("results", []):
    cfg  = entry.get("config", {})
    agg  = entry.get("aggregate", {})
    hdim = cfg.get("hidden_dim", "?")
    tpr  = cfg.get("tokens_per_rank", "?")
    mode = cfg.get("routing_mode", "?")
    p50  = agg.get("comm_share_p50_worst_rank", float("nan"))
    p95  = agg.get("comm_share_p95_worst_rank", float("nan"))
    dec  = agg.get("decision", decision_str(p50))
    print(f"{hdim:>10} {tpr:>12} {mode:>15} {p50:>9.1%} {p95:>9.1%} {dec:>12}")

## Phase 4 Gate Report — Final Aggregation

Runs `phase4_gate_report.py --strict-phase4` over all three JSON outputs.  
This is the canonical go/warn/stop decision for Phase 4 completion.

In [ ]:
GATE_REPORT_JSON = "results/phase4/phase4_gate_report_kaggle.json"

gate_cmd = [
    sys.executable, "experiments/phase4_gate_report.py",
    "--baseline-json",   BASELINE_JSON,
    "--parity-json",     PARITY_JSON,
    "--microbench-json", MICRO_JSON,
    "--strict-phase4",
    "--output",          GATE_REPORT_JSON,
]

rc = run_cmd(gate_cmd, label="GATE-REPORT")
assert rc == 0, f"Gate report script exited with code {rc}"

# ── Final Phase 4 verdict ──────────────────────────────────────────────────────
report = load_json(GATE_REPORT_JSON)
overall = report.get("overall", {})
status  = overall.get("status", "unknown").upper()
summary = overall.get("summary", "")
blocking = overall.get("blocking_gates", [])
warnings = overall.get("warning_gates", [])

BANNER = {
    "GO":   ("=" * 60, "  ★  PHASE 4 GO — ALL GATES PASS                      "),
    "WARN": ("=" * 60, "  ⚠   PHASE 4 WARN — CHECK WARNING GATES               "),
    "STOP": ("=" * 60, "  ✗   PHASE 4 STOP — BLOCKED GATES MUST BE RESOLVED    "),
}
sep, headline = BANNER.get(status, ("=" * 60, f"  STATUS: {status}"))

print(sep)
print(headline)
print(sep)
print(f"\nSummary : {summary}")
if blocking:
    print(f"Blocking: {', '.join(blocking)}")
if warnings:
    print(f"Warnings: {', '.join(warnings)}")

print("\n── Per-gate details ─────────────────────────────────────────")
for gate in report.get("gates", []):
    g_name   = gate.get("name", "?")
    g_status = gate.get("status", "?").upper()
    g_sum    = gate.get("summary", "")
    icon     = "✓" if g_status == "GO" else ("⚠" if g_status == "WARN" else "✗")
    print(f"  {icon}  [{g_status:<4}] {g_name:<20}  {g_sum}")
    for note in gate.get("notes", []):
        print(f"           note: {note}")

print("\n── Raw JSON ─────────────────────────────────────────────────")
pp(report)

# ── Archive all result JSONs to /kaggle/working so Kaggle saves them ───────────

OUTPUT_DIR = Path("/kaggle/working/phase4_gate_results")
OUTPUT_DIR.mkdir(exist_ok=True)

for src in [BASELINE_JSON, PARITY_JSON, MICRO_JSON, GATE_REPORT_JSON]:
    dst = OUTPUT_DIR / Path(src).name
    shutil.copy2(src, dst)
    print(f"Saved: {dst}")

print(f"\nAll results archived to {OUTPUT_DIR}")
print("Download via Kaggle → Output tab, or use the Kaggle API:")
print(f"  kaggle kernels output <your-kernel-slug>")